In [ ]:
from pathlib import Path
Path("docs").mkdir(exist_ok=True)
Path("docs/doc_01.txt").write_text("Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.", encoding="utf-8")
Path("docs/doc_02.txt").write_text("grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.", encoding="utf-8")
Path("docs/doc_03.txt").write_text("Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.", encoding="utf-8")
Path("docs/doc_04.txt").write_text("Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.", encoding="utf-8")
Path("docs/doc_05.txt").write_text("Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.", encoding="utf-8")
Path("docs/doc_06.txt").write_text("If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.", encoding="utf-8")
Path("docs/doc_07.txt").write_text("Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.", encoding="utf-8")
Path("docs/doc_08.txt").write_text("Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered.", encoding="utf-8")
print("created 8 documents succcessfully.")

created 8 documents succcessfully.


In [ ]:
!find /content -type f | head -50

/content/.config/default_configs.db
/content/.config/active_config
/content/.config/gce
/content/.config/.last_opt_in_prompt.yaml
/content/.config/configurations/config_default
/content/.config/.last_update_check.json
/content/.config/logs/2026.09.16/13.26.04.283691.log
/content/.config/logs/2026.09.16/13.25.37.877256.log
/content/.config/logs/2026.09.16/13.26.29.139725.log
/content/.config/logs/2026.09.16/13.26.15.129583.log
/content/.config/logs/2026.09.16/13.26.17.091876.log
/content/.config/logs/2026.09.16/13.26.30.459687.log
/content/.config/config_sentinel
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/.last_survey_prompt.yaml
/content/sample_data/anscombe.json
/content/sample_data/README.md
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv


In [ ]:
! find /content/drive -type f -path "*/docs/*" 2>/dev/null | head -30

In [ ]:
!mkdir -p docs

In [ ]:
from pydantic import BaseModel, Field, field_validator


class AskRequest(BaseModel):
    query: str


class AnswerResponse(BaseModel):
    answer: str
    sources: list[str] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float) -> float:
        if not 0.0 <= value <= 1.0:
            raise ValueError("confidence must be between 0 and 1")
        return value

In [ ]:
STRUCTURED_PROMPT = """
ROLE
You are Zepto's customer support assistant. Answer customer questions
accurately using only the supplied Zepto policy context.

CONTEXT
The retrieved context below contains excerpts from Zepto policy documents.

{context}

TASK
Answer the customer's question using the retrieved context.

Customer question:
{query}

FORMAT
Return a JSON object with exactly these fields:
{
  "answer": "string",
  "sources": ["chunk/document IDs"],
  "confidence": 0.0
}

The confidence value must be a number between 0 and 1.
The sources field must contain only IDs of context chunks actually used.

LENGTH
Keep the answer concise: normally 1–3 sentences.

NEGATIVE CONSTRAINTS
- Do not answer using information not present in the provided context.
- Do not invent Zepto policies, prices, deadlines, or procedures.
- If the context does not contain enough information to answer the question,
  say that the supplied policy context does not contain the required information.
- Do not include Markdown outside the JSON object.

FEW-SHOT EXAMPLE

Example context:
[doc_example] Zepto standard delivery is free on orders over INR 149.

Example question:
Is standard delivery free for an order above INR 149?

Example answer:
{
  "answer": "Yes. Standard delivery is free on orders over INR 149.",
  "sources": ["doc_example"],
  "confidence": 1.0
}

Now answer the actual customer question using only the supplied context.
"""

In [ ]:
%pip install -q chromadb sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [ ]:
import chromadb

In [ ]:
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer


BASE_DIR = Path("/content")
DOCS_DIR = BASE_DIR / "docs"
CHROMA_DIR = BASE_DIR / "chroma_db"

COLLECTION_NAME = "zepto_policy"

MODEL_NAME = "all-MiniLM-L6-v2"


def get_embedding_model():
    return SentenceTransformer(MODEL_NAME)


def get_collection():
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

    return collection


def load_documents():
    documents = []
    ids = []

    for path in sorted(DOCS_DIR.glob("doc_*.txt")):
        documents.append(path.read_text(encoding="utf-8").strip())
        ids.append(path.stem)

    if len(documents) != 8:
        raise RuntimeError(
            f"Expected exactly 8 corpus documents, found {len(documents)}"
        )

    return ids, documents


def ingest():
    ids, documents = load_documents()

    model = get_embedding_model()
    embeddings = model.encode(
        documents,
        normalize_embeddings=True,
    ).tolist()

    collection = get_collection()

    # Re-running ingestion is safe and deterministic.
    collection.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=[
            {"document_id": doc_id}
            for doc_id in ids
        ],
    )

    print(f"Ingested {collection.count()} chunks into {COLLECTION_NAME}")
    print(f"Documents: {ids}")


if __name__ == "__main__":
    ingest()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ingested 8 chunks into zepto_policy
Documents: ['doc_01', 'doc_02', 'doc_03', 'doc_04', 'doc_05', 'doc_06', 'doc_07', 'doc_08']


In [3]:
!pip install -q langchain-groq

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("/content/langchain-groq")))

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/content")
print("Python path updated")

Python path updated


In [ ]:
!find /content -name "ingest.py" -type f

In [ ]:
!find /content -maxdepth 3 -type f | head -50

/content/.config/default_configs.db
/content/.config/active_config
/content/.config/gce
/content/.config/.last_opt_in_prompt.yaml
/content/.config/configurations/config_default
/content/.config/.last_update_check.json
/content/.config/config_sentinel
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/.last_survey_prompt.yaml
/content/docs/doc_02.txt
/content/docs/doc_03.txt
/content/docs/doc_04.txt
/content/docs/doc_08.txt
/content/docs/doc_07.txt
/content/docs/doc_05.txt
/content/docs/doc_06.txt
/content/docs/doc_01.txt
/content/chroma_db/c17ca431-a771-4c0a-98ce-db30c9dc5e28/header.bin
/content/chroma_db/c17ca431-a771-4c0a-98ce-db30c9dc5e28/length.bin
/content/chroma_db/c17ca431-a771-4c0a-98ce-db30c9dc5e28/link_lists.bin
/content/chroma_db/c17ca431-a771-4c0a-98ce-db30c9dc5e28/data_level0.bin
/content/chroma_db/chroma.sqlite3
/content/sample_data/anscombe.json
/content/sample_data/README.md
/content/sample_data/mnist_train_small.csv
/conten

In [ ]:
%%writefile /content/ingest.py
def get_collection():
  return None
def get_embedding_model():
  return none

Writing /content/ingest.py


In [ ]:
%%writefile /content/models.py
from pydantic import BaseModel

class AnswerResponse(BaseModel):
    answer: str


Overwriting /content/models.py


In [ ]:
%%writefile /content/prompts.py
STRUCTURED_PROMPT = """
ROLE
You are Zepto's customer support assistant. Answer customer questions
accurately using only the supplied Zepto policy context.

Writing /content/prompts.py


In [ ]:
import json
import os
from typing import TypedDict

In [ ]:
%%writefile /content/graph.py
def run_query(quetion):
  return{"answer": "zepto support assistant is ready.", "quetion": quetion}

Writing /content/graph.py


In [ ]:
%%writefile /content/models.py
from pydantic import BaseModel

class AnswerResponse(BaseModel):
    answer: str
class AskRequest(BaseModel):
    query: str

Overwriting /content/models.py


In [ ]:
from ingest import get_collection, get_embedding_model
from models import AnswerResponse
from prompts import STRUCTURED_PROMPT

SyntaxError: unterminated triple-quoted string literal (detected at line 4) (prompts.py, line 1)

In [5]:
STRUCTURED_PROMT = """
Classify the customer query into exactly one category:

policy_question
general_question

Use policy_question when the question concerns a Zepto policy,
delivery, returns, refunds, membership, tracking, cancellation,
gift cards, or support.

Use general_question otherwise.

Return only the category name,
Query:{query}"""

In [7]:
import json
import os
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from ingest import get_collection, get_embedding_model
from models import AnswerResponse
from prompts import STRUCTURED_PROMPT


POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours",
]


class SupportState(TypedDict, total=False):
    query: str
    intent: str
    retrieved_ids: list[str]
    retrieved_documents: list[str]
    answer: str
    sources: list[str]
    confidence: float
    error: str


def mock_enabled() -> bool:
    """
    MOCK_LLM is enabled unless explicitly set to 0.
    """
    return os.getenv("MOCK_LLM", "1") != "0"


def classify_mock(query: str) -> str:
    query_lower = query.lower()

    if any(keyword in query_lower for keyword in POLICY_KEYWORDS):
        return "policy_question"

    return "general_question"


def get_real_llm():
    """
    Optional extension only.

    The API key is read from GROQ_API_KEY and is never hardcoded.
    """
    return ChatGroq(
        model=os.getenv(
            "GROQ_MODEL",
            "llama-3.1-8b-instant",
        ),
        temperature=0,
        api_key=os.environ["GROQ_API_KEY"],
    )


def classify_intent(state: SupportState) -> SupportState:
    query = state["query"]

    if mock_enabled():
        intent = classify_mock(query)
    else:
        llm = get_real_llm()

        prompt = f"""
Classify the following customer query into exactly one category:

policy_question
general_question

Use policy_question when the question concerns a Zepto policy,
delivery, returns, refunds, membership, tracking, cancellation,
gift cards, or support.

Use general_question otherwise.

Return only the category name.

Query:
{query}
"""

        response = llm.invoke(prompt)
        intent = response.content.strip()

        if intent not in {"policy_question", "general_question"}:
            intent = "general_question"

    return {
        **state,
        "intent": intent,
    }


def retrieve_documents(query: str):
    model = get_embedding_model()
    collection = get_collection()

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
    )[0].tolist()

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "metadatas", "distances"],
    )

    ids = result["ids"][0]
    documents = result["documents"][0]

    return ids, documents


def build_mock_answer(document: str) -> str:
    snippet = document[:200].strip()

    return f"Based on the retrieved context: {snippet}"


def parse_llm_response(raw: str) -> AnswerResponse:
    """
    Parse and validate the structured response returned by the optional
    real LLM branch.
    """
    # Handle accidental Markdown code fences.
    cleaned = raw.strip()

    if cleaned.startswith("```"):
        lines = cleaned.splitlines()

        if lines and lines[0].startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        cleaned = "\n".join(lines).strip()

    data = json.loads(cleaned)
    return AnswerResponse.model_validate(data)


def real_llm_answer(
    query: str,
    documents: list[str],
    ids: list[str],
) -> AnswerResponse:
    llm = get_real_llm()

    context_parts = []

    for chunk_id, document in zip(ids, documents):
        context_parts.append(
            f"[{chunk_id}]\n{document}"
        )

    context = "\n\n".join(context_parts)

    prompt = STRUCTURED_PROMPT.format(
        context=context,
        query=query,
    )

    last_error = None

    # Initial attempt + up to 2 corrective retries.
    for attempt in range(3):
        if attempt == 0:
            current_prompt = prompt
        else:
            current_prompt = f"""
Your previous response failed JSON/schema validation.

Return ONLY valid JSON matching this exact schema:

{{
  "answer": "string",
  "sources": ["string"],
  "confidence": 0.0
}}

Do not add Markdown fences or explanatory text.

Original task:
{prompt}
"""

        try:
            response = llm.invoke(current_prompt)
            return parse_llm_response(response.content)

        except Exception as exc:
            last_error = exc

    return AnswerResponse(
        answer=(
            "ERROR: The real LLM response could not be validated "
            "against the required response schema."
        ),
        sources=[],
        confidence=0.0,
    )


def retrieve_and_answer(state: SupportState) -> SupportState:
    query = state["query"]

    # Retrieval ALWAYS happens, regardless of MOCK_LLM.
    ids, documents = retrieve_documents(query)

    if mock_enabled():
        answer = build_mock_answer(documents[0])

        response = AnswerResponse(
            answer=answer,
            sources=ids,
            confidence=1.0,
        )

    else:
        response = real_llm_answer(
            query=query,
            documents=documents,
            ids=ids,
        )

    return {
        **state,
        "retrieved_ids": ids,
        "retrieved_documents": documents,
        "answer": response.answer,
        "sources": response.sources,
        "confidence": response.confidence,
    }


def direct_answer(state: SupportState) -> SupportState:
    query = state["query"]

    if mock_enabled():
        response = AnswerResponse(
            answer=(
                "I can only answer questions about Zepto policies right now."
            ),
            sources=[],
            confidence=1.0,
        )

    else:
        llm = get_real_llm()

        prompt = f"""
You are Zepto's customer support assistant.

Answer the customer's question directly.

Return JSON with exactly:
{{
  "answer": "string",
  "sources": [],
  "confidence": 0.0
}}

Do not invent Zepto policy information.

Question:
{query}
"""

        last_error = None

        for attempt in range(3):
            try:
                response = llm.invoke(prompt)

                parsed = parse_llm_response(response.content)

                # Direct questions must not claim retrieved sources.
                parsed = AnswerResponse(
                    answer=parsed.answer,
                    sources=[],
                    confidence=parsed.confidence,
                )

                response = parsed
                break

            except Exception as exc:
                last_error = exc

                prompt = f"""
The previous response failed schema validation.

Return ONLY valid JSON:
{{
  "answer": "string",
  "sources": [],
  "confidence": 0.0
}}

Question:
{query}
"""
        else:
            response = AnswerResponse(
                answer=(
                    "ERROR: The real LLM response could not be validated "
                    "against the required response schema."
                ),
                sources=[],
                confidence=0.0,
            )

    return {
        **state,
        "answer": response.answer,
        "sources": response.sources,
        "confidence": response.confidence,
    }


def route_after_classification(state: SupportState) -> str:
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"

    return "direct_answer"


def build_graph():
    graph = StateGraph(SupportState)

    graph.add_node("classify_intent", classify_intent)
    graph.add_node("retrieve_and_answer", retrieve_and_answer)
    graph.add_node("direct_answer", direct_answer)

    graph.set_entry_point("classify_intent")

    graph.add_conditional_edges(
        "classify_intent",
        route_after_classification,
        {
            "retrieve_and_answer": "retrieve_and_answer",
            "direct_answer": "direct_answer",
        },
    )

    graph.add_edge("retrieve_and_answer", END)
    graph.add_edge("direct_answer", END)

    return graph.compile()


support_graph = build_graph()


def run_query(query: str) -> AnswerResponse:
    state = support_graph.invoke(
        {
            "query": query,
        }
    )

    response = AnswerResponse(
        answer=state["answer"],
        sources=state.get("sources", []),
        confidence=state.get("confidence", 0.0),
    )

    # Final application-level validation.
    return AnswerResponse.model_validate(
        response.model_dump()
    )

ModuleNotFoundError: No module named 'ingest'

In [ ]:
from fastapi import FastAPI

from graph import run_query
from models import AskRequest, AnswerResponse

In [ ]:
%%writefile /content/models.py
from pydantic import BaseModel


class AskRequest(BaseModel):
    query: str


class AnswerResponse(BaseModel):
    answer: str
    quetion: str


Writing /content/models.py


In [ ]:
from fastapi import FastAPI

from graph import run_query
from models import AskRequest, AnswerResponse


app = FastAPI(
    title="Zepto Support Assistant",
    version="1.0.0",
)


@app.get("/")
def root():
    return {
        "service": "Zepto Support Assistant",
        "endpoint": "POST /ask",
        "mock_llm": "enabled by default",
    }


@app.post("/ask", response_model=AnswerResponse)
def ask(request: AskRequest) -> AnswerResponse:
    return run_query(request.query)

In [ ]:
{
  "answer": "string",
  "sources": ["chunk/document IDs"],
  "confidence": 0.0
}

{'answer': 'string', 'sources': ['chunk/document IDs'], 'confidence': 0.0}

In [4]:
export MOCK_LLM=0
export GROQ_API_KEY="your-key"

SyntaxError: invalid syntax (1589693025.py, line 1)

# Zepto Support Assistant

Module 3 implementation: `/support_assistant`

This project implements a small Retrieval-Augmented Generation (RAG)
support service for Zepto using:

- Sentence Transformers with `all-MiniLM-L6-v2`
- ChromaDB
- LangGraph
- Pydantic
- FastAPI
- Uvicorn
- An offline deterministic mock LLM mode